# Zakuro — QUIC transport resilience

Demonstrates that `QuicProcessor` recovers from transient connection failures without the caller knowing. Three scenarios:

1. **Baseline** — a plain `@zk.fn` dispatch over QUIC (~100 ms).
2. **In-flight during SIGKILL** — the worker disappears silently. With the default `idle_timeout` (aioquic: 30–60 s) the caller would hang for a minute; Zakuro sets it to **5 s**, so the failure surfaces quickly as `ConnectionError`.
3. **After respawn** — the worker comes back on the same port. The next dispatch builds a fresh connection and returns in baseline time.

Every timing is measured from the live subprocess lifecycle.

In [ ]:
import signal
import subprocess
import time

import zakuro as zk
from zakuro.worker.runner import _locate_zakuro_worker

print("zakuro:", zk.__version__)

## Helper — spawn a QUIC worker on a known port

`zk.Worker.spawn()` picks an ephemeral port. For this demo we want a fixed port so we can bounce the worker and rebind to the same URI.

In [ ]:
PORT = 4460

def spawn_quic(port: int, name: str) -> subprocess.Popen:
    binary = _locate_zakuro_worker()
    proc = subprocess.Popen(
        [binary, "--transport", "quic", "--host", "127.0.0.1",
         "--port", str(port), "--worker-name", name],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    # Poll HEALTH until ready (or the process dies).
    from zakuro.compute import Compute
    from zakuro.processors.base import ProcessorConfig
    from zakuro.processors.quic import QuicProcessor

    deadline = time.perf_counter() + 15.0
    while time.perf_counter() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(f"worker {name!r} exited early rc={proc.returncode}")
        try:
            compute = Compute(uri=f"quic://127.0.0.1:{port}", verify=False)
            cfg = ProcessorConfig(scheme="quic", host="127.0.0.1", port=port)
            p = QuicProcessor(cfg, compute)
            p.connect()
            try:
                if p.ping():
                    return proc
            finally:
                p.disconnect()
        except Exception:
            pass
        time.sleep(0.2)
    proc.terminate()
    raise TimeoutError(f"worker {name!r} not healthy within 15 s")

proc = spawn_quic(PORT, "quic-demo")
print(f"worker up, pid={proc.pid}, port={PORT}")

## 1. Baseline dispatch

`zk.Compute(uri="quic://127.0.0.1:PORT")` routes the call through `QuicProcessor`. Timing measured from before `.to()` to the returned result.

In [ ]:
compute = zk.Compute(uri=f"quic://127.0.0.1:{PORT}", verify=False)

@zk.fn
def add(a: int, b: int) -> int:
    return a + b

t0 = time.perf_counter()
result = add.to(compute)(41, 1)
baseline_ms = 1000 * (time.perf_counter() - t0)
print(f"baseline result: {result}   ({baseline_ms:.1f} ms)")

## 2. In-flight dispatch during SIGKILL

Kill the worker, then try a dispatch. The worker is gone but the client's socket doesn't know — only aioquic's idle timer notices. Without Zakuro's 5 s override, the default aioquic idle_timeout is 30–60 s. Here, it fails in ~5 s.

In [ ]:
print("SIGKILL")
proc.send_signal(signal.SIGKILL)
proc.wait()

t0 = time.perf_counter()
try:
    add.to(compute)(99, 1)
    outcome = "unexpectedly succeeded"
except Exception as exc:
    outcome = f"failed: {type(exc).__name__}"
detection_secs = time.perf_counter() - t0
print(f"dispatch during kill: {outcome} after {detection_secs:.1f} s")

## 3. Respawn on the same port, dispatch again

Same URI, fresh subprocess. The retry path inside `QuicProcessor.execute` tears down any stale connection and dials again on failure, but in this specific case the `zk.Compute` is fresh per call (`Fn._execute_single_compute` constructs a new processor), so the call simply succeeds on first attempt.

In [ ]:
proc2 = spawn_quic(PORT, "quic-demo-2")
print(f"respawned, pid={proc2.pid}")

t0 = time.perf_counter()
result = add.to(compute)(100, 1)
post_ms = 1000 * (time.perf_counter() - t0)
print(f"post-respawn dispatch: {result}   ({post_ms:.1f} ms)")

## Summary

Printed together so the three numbers are easy to read.

In [ ]:
print(f"baseline              : {baseline_ms:7.1f} ms")
print(f"dispatch during kill  : {detection_secs*1000:7.1f} ms  (aioquic default would be 30–60 s)")
print(f"dispatch post-respawn : {post_ms:7.1f} ms")

## Cleanup

In [ ]:
proc2.terminate()
proc2.wait()
print("done")